<a href="https://colab.research.google.com/github/rkikkeri/ML-DL/blob/main/Copy_of_FAc_fondparinux_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# compound_x_hs_probe_single_file.py
# Install:
# pip install openai transformers torch scikit-learn pandas numpy openpyxl
# Optional for SMILES descriptors:
# conda install -c conda-forge rdkit
# or: pip install rdkit

import os, json, warnings
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import torch


In [12]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
    RDKIT_AVAILABLE = True
except Exception:
    RDKIT_AVAILABLE = False

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except Exception:
    OPENAI_AVAILABLE = False


COMPOUND_X = {
    "name": "Compound X",
    "smiles": "O[C@H]1[C@H](O)[C@@H](NC(C)=O)[C@@H](O[C@@H]([C@H](F)[C@@H](O)[C@H](O[C@@H]([C@H](OSOO[O-])[C@@H](NC(C)=O)[C@@H](O[C@@H]([C@H](O)[C@H]2OSOO[O-])[C@H](C(O)=O)O[C@H]2O[C@H]3C(O)[C@@H](NC(C)=O)[C@@H](OC)O[C@@H]3COS(=O)([O-])=O)O4)[C@H]4COS(=O)([O-])=O)O5)[C@H]5C(O)=O)O[C@@H]1COS(=O)([O-])=O",
    "description": """
    Fondaparinux mimic / synthetic heparan sulfate probe.
    Add exact sequence, sulfation pattern, fluorination, linker, hydrophobic tag,
    ring conformation, and any biological assay data.
    """,
    "num_sugars": 5,
    "n_sulfate": 0,
    "o2_sulfate": 1,
    "o3_sulfate": 1,
    "o6_sulfate": 3,
    "carboxylates": 2,
    "has_fondaparinux_like_ATIII_motif": True,
    "has_fluorination": True,
    "has_hydrophobic_tag": False,
    "known_anti_Xa_activity": None
}


In [13]:

HS_BINDING_PROTEINS = [
    {"protein": "Antithrombin III", "gene": "SERPINC1", "application": "Anticoagulant / anti-Xa", "keywords": "fondaparinux antithrombin heparin pentasaccharide 3-O-sulfate factor Xa"},
    {"protein": "Factor Xa", "gene": "F10", "application": "Anticoagulant / anti-Xa", "keywords": "factor Xa anticoagulation fondaparinux antithrombin"},
    {"protein": "PF4", "gene": "PF4", "application": "Platelet / HIT risk", "keywords": "platelet factor 4 heparin HIT immune complex thrombocytopenia"},
    {"protein": "FGF1", "gene": "FGF1", "application": "FGF signaling", "keywords": "FGF1 heparan sulfate growth factor FGFR signaling"},
    {"protein": "FGF2", "gene": "FGF2", "application": "FGF signaling / angiogenesis", "keywords": "FGF2 heparan sulfate FGFR dimerization angiogenesis proliferation"},
    {"protein": "FGFR1", "gene": "FGFR1", "application": "FGF receptor signaling", "keywords": "FGFR1 heparan sulfate FGF receptor signaling"},
    {"protein": "VEGF165", "gene": "VEGFA", "application": "Angiogenesis", "keywords": "VEGF165 heparan sulfate angiogenesis endothelial binding"},
    {"protein": "HGF", "gene": "HGF", "application": "Cancer / regeneration", "keywords": "HGF heparan sulfate c-Met growth factor cancer metastasis"},
    {"protein": "BMP2", "gene": "BMP2", "application": "BMP signaling", "keywords": "BMP2 heparan sulfate SMAD differentiation morphogen cancer ALS"},
    {"protein": "BMP4", "gene": "BMP4", "application": "BMP signaling", "keywords": "BMP4 heparan sulfate SMAD morphogen differentiation breast cancer"},
    {"protein": "BMP7", "gene": "BMP7", "application": "BMP signaling / neurobiology", "keywords": "BMP7 heparan sulfate neurobiology fibrosis regeneration"},
    {"protein": "Wnt3a", "gene": "WNT3A", "application": "Wnt signaling", "keywords": "Wnt3a heparan sulfate glypican Frizzled LRP6 stem cell cancer"},
    {"protein": "Wnt5a", "gene": "WNT5A", "application": "Noncanonical Wnt signaling", "keywords": "Wnt5a heparan sulfate glypican cancer inflammation"},
    {"protein": "CXCL8 / IL-8", "gene": "CXCL8", "application": "Inflammation / chemokine", "keywords": "IL8 CXCL8 heparan sulfate chemokine neutrophil inflammation"},
    {"protein": "CXCL12 / SDF1", "gene": "CXCL12", "application": "Chemokine / metastasis", "keywords": "CXCL12 SDF1 heparan sulfate CXCR4 migration metastasis stem cell"},
    {"protein": "CCL2 / MCP1", "gene": "CCL2", "application": "Inflammation", "keywords": "CCL2 MCP1 heparan sulfate monocyte chemokine inflammation"},
    {"protein": "Heparanase", "gene": "HPSE", "application": "Cancer metastasis / ECM remodeling", "keywords": "heparanase heparan sulfate cleavage tumor invasion metastasis inhibitor"},
    {"protein": "L-selectin", "gene": "SELL", "application": "Leukocyte adhesion", "keywords": "selectin heparan sulfate leukocyte rolling adhesion inflammation"},
    {"protein": "P-selectin", "gene": "SELP", "application": "Platelet / inflammation", "keywords": "P-selectin heparan sulfate platelet adhesion inflammation metastasis"},
    {"protein": "Galectin-3", "gene": "LGALS3", "application": "Cancer / immune modulation", "keywords": "galectin 3 heparan sulfate cancer immune fibrosis"},
    {"protein": "SARS-CoV-2 Spike RBD", "gene": "SPIKE", "application": "Antiviral HS mimic", "keywords": "viral spike heparan sulfate attachment entry antiviral mimic"},
    {"protein": "HSV glycoprotein D", "gene": "gD", "application": "Antiviral HS mimic", "keywords": "herpes glycoprotein D heparan sulfate viral attachment entry"},
    {"protein": "Tau", "gene": "MAPT", "application": "Neurobiology", "keywords": "tau heparan sulfate aggregation neurodegeneration uptake"},
    {"protein": "ApoE", "gene": "APOE", "application": "Lipid / neurobiology", "keywords": "ApoE heparan sulfate lipoprotein neurobiology Alzheimer's"}
]


In [14]:

def smiles_descriptors(smiles):
    out = {
        "rdkit_available": RDKIT_AVAILABLE,
        "valid_smiles": False,
        "mol_weight": None,
        "logp": None,
        "tpsa": None,
        "hbd": None,
        "hba": None,
        "rotatable_bonds": None,
        "ring_count": None
    }

    if not smiles or smiles == "PUT_YOUR_SMILES_HERE":
        return out

    if not RDKIT_AVAILABLE:
        return out

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return out

    out.update({
        "valid_smiles": True,
        "mol_weight": round(Descriptors.MolWt(mol), 2),
        "logp": round(Crippen.MolLogP(mol), 2),
        "tpsa": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "hbd": Descriptors.NumHDonors(mol),
        "hba": Descriptors.NumHAcceptors(mol),
        "rotatable_bonds": Descriptors.NumRotatableBonds(mol),
        "ring_count": rdMolDescriptors.CalcNumRings(mol)
    })
    return out



In [15]:

def hs_structural_features(c):
    total_sulfates = c["n_sulfate"] + c["o2_sulfate"] + c["o3_sulfate"] + c["o6_sulfate"]
    total_negative_charge = total_sulfates + c["carboxylates"]
    sulfate_density = total_sulfates / max(c["num_sugars"], 1)

    anticoagulant_score = 0
    anticoagulant_score += 0.35 if c["has_fondaparinux_like_ATIII_motif"] else 0
    anticoagulant_score += 0.25 if c["o3_sulfate"] >= 1 else 0
    anticoagulant_score += 0.15 if c["num_sugars"] == 5 else 0
    anticoagulant_score += 0.15 if total_negative_charge >= 7 else 0
    anticoagulant_score += 0.10 if c["known_anti_Xa_activity"] else 0

    broad_probe_score = 0
    broad_probe_score += 0.25 if total_negative_charge >= 6 else 0
    broad_probe_score += 0.20 if sulfate_density >= 1.0 else 0
    broad_probe_score += 0.15 if c["o6_sulfate"] >= 2 else 0
    broad_probe_score += 0.15 if c["o2_sulfate"] >= 1 else 0
    broad_probe_score += 0.10 if c["has_fluorination"] else 0
    broad_probe_score += 0.10 if c["has_hydrophobic_tag"] else 0
    broad_probe_score += 0.05 if c["num_sugars"] >= 5 else 0

    return {
        "total_sulfates": total_sulfates,
        "total_negative_charge": total_negative_charge,
        "sulfate_density": round(sulfate_density, 2),
        "anticoagulant_feature_score": round(min(anticoagulant_score, 1.0), 2),
        "broad_hs_probe_feature_score": round(min(broad_probe_score, 1.0), 2)
    }


class BertEmbedder:
    def __init__(self, model_name="allenai/scibert_scivocab_uncased"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

    def embed(self, texts):
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )
        with torch.no_grad():
            output = self.model(**encoded)
        return output.last_hidden_state[:, 0, :].numpy()



In [16]:

def predict_targets(compound, protein_panel):
    smiles_feat = smiles_descriptors(compound["smiles"])
    hs_feat = hs_structural_features(compound)

    compound_text = f"""
    {compound["name"]}
    SMILES: {compound["smiles"]}
    Description: {compound["description"]}
    HS features: {json.dumps(hs_feat)}
    SMILES descriptors: {json.dumps(smiles_feat)}
    """

    model = BertEmbedder()
    compound_emb = model.embed([compound_text])
    protein_texts = [
        f'{p["protein"]} {p["gene"]} {p["application"]} {p["keywords"]}'
        for p in protein_panel
    ]
    protein_embs = model.embed(protein_texts)

    sims = cosine_similarity(compound_emb, protein_embs)[0]

    rows = []
    for p, sim in zip(protein_panel, sims):
        structural_bonus = hs_feat["broad_hs_probe_feature_score"]

        if p["application"].lower().startswith("anticoagulant"):
            structural_bonus = hs_feat["anticoagulant_feature_score"]
        elif "BMP" in p["application"]:
            structural_bonus += 0.08 if hs_feat["total_negative_charge"] >= 7 else 0
        elif "FGF" in p["application"]:
            structural_bonus += 0.08 if compound["o6_sulfate"] >= 2 else 0
        elif "Chemokine" in p["application"] or "Inflammation" in p["application"]:
            structural_bonus += 0.06 if hs_feat["total_sulfates"] >= 5 else 0
        elif "Heparanase" in p["protein"]:
            structural_bonus += 0.08 if hs_feat["total_sulfates"] >= 5 else 0
        elif "Wnt" in p["application"]:
            structural_bonus += 0.06 if compound["has_hydrophobic_tag"] or compound["has_fluorination"] else 0

        structural_bonus = min(structural_bonus, 1.0)
        final_score = round(0.55 * float(sim) + 0.45 * structural_bonus, 3)

        if final_score >= 0.78:
            confidence = "High"
        elif final_score >= 0.68:
            confidence = "Medium"
        else:
            confidence = "Exploratory"

        rows.append({
            "Compound": compound["name"],
            "Protein": p["protein"],
            "Gene": p["gene"],
            "Predicted application": p["application"],
            "BERT similarity": round(float(sim), 3),
            "Structural HS score": round(structural_bonus, 2),
            "Final priority score": final_score,
            "Confidence": confidence
        })

    df = pd.DataFrame(rows).sort_values("Final priority score", ascending=False)
    return df, hs_feat, smiles_feat


In [17]:
def summarize_by_application(target_df):
    grouped = (
        target_df
        .groupby("Predicted application")
        .agg(
            Mean_score=("Final priority score", "mean"),
            Best_score=("Final priority score", "max"),
            Top_protein=("Protein", "first"),
            Confidence=("Confidence", "first")
        )
        .reset_index()
        .sort_values("Best_score", ascending=False)
    )
    return grouped



In [28]:


def openai_summary(compound, hs_features, smiles_features, target_df):
    if not OPENAI_AVAILABLE or not os.environ.get("gl-U2FsdGVkX18czT8etifeiSSve1sfN2HNEey+qDfBDJmmP2jwwQ6GdW8lT7ScyMBz"):
        return "OpenAI summary skipped: OPENAI_API_KEY not found or openai package not installed."

    client = OpenAI()

    top_targets = target_df.head(12).to_dict(orient="records")

    prompt = f"""
You are an expert in heparan sulfate biology, synthetic glycochemistry, anticoagulants,
glycan-protein interactions, cancer biology, inflammation, and neurobiology.

Compound:
{json.dumps(compound, indent=2)}

HS structural features:
{json.dumps(hs_features, indent=2)}

SMILES/RDKit descriptors:
{json.dumps(smiles_features, indent=2)}

Top predicted targets:
{json.dumps(top_targets, indent=2)}

Write a concise scientific interpretation with:
1. Whether Compound X should be positioned as selective synthetic HS probe, not only anticoagulant.
2. Top 5 biological applications.
3. Why anticoagulant activity may or may not dominate.
4. Key validation experiments.
5. Best grant-style hypothesis.
"""

    response = client.responses.create(
        model="gpt-5.2",
        input=prompt
    )
    return response.output_text


In [29]:
def write_excel(compound, target_df, app_df, hs_features, smiles_features, llm_text):
    file_name = f"{compound['name'].replace(' ', '_')}_HS_probe_prediction.xlsx"

    compound_df = pd.DataFrame([compound])
    hs_df = pd.DataFrame([hs_features])
    smiles_df = pd.DataFrame([smiles_features])
    llm_df = pd.DataFrame([{"OpenAI interpretation": llm_text}])

    with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
        compound_df.to_excel(writer, sheet_name="Compound_Input", index=False)
        hs_df.to_excel(writer, sheet_name="HS_Features", index=False)
        smiles_df.to_excel(writer, sheet_name="SMILES_Descriptors", index=False)
        target_df.to_excel(writer, sheet_name="Protein_Target_Ranking", index=False)
        app_df.to_excel(writer, sheet_name="Application_Summary", index=False)
        llm_df.to_excel(writer, sheet_name="OpenAI_Interpretation", index=False)

    return file_name



In [30]:

def main():
    print("Running Compound X HS-probe prediction...")

    target_df, hs_features, smiles_features = predict_targets(COMPOUND_X, HS_BINDING_PROTEINS)
    app_df = summarize_by_application(target_df)

    llm_text = openai_summary(COMPOUND_X, hs_features, smiles_features, target_df)

    excel_file = write_excel(
        COMPOUND_X,
        target_df,
        app_df,
        hs_features,
        smiles_features,
        llm_text
    )

    print("\nTop predicted protein targets:")
    print(target_df.head(10).to_string(index=False))

    print("\nTop predicted applications:")
    print(app_df.head(10).to_string(index=False))

    print(f"\nExcel file written: {excel_file}")


if __name__ == "__main__":
    main()

Running Compound X HS-probe prediction...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Top predicted protein targets:
  Compound          Protein     Gene              Predicted application  BERT similarity  Structural HS score  Final priority score Confidence
Compound X             BMP2     BMP2                      BMP signaling            0.777                 0.98                 0.868       High
Compound X       Heparanase     HPSE Cancer metastasis / ECM remodeling            0.669                 0.98                 0.809       High
Compound X      CCL2 / MCP1     CCL2                       Inflammation            0.682                 0.96                 0.807       High
Compound X             ApoE     APOE               Lipid / neurobiology            0.727                 0.90                 0.805       High
Compound X             BMP4     BMP4                      BMP signaling            0.654                 0.98                 0.800       High
Compound X             FGF2     FGF2       FGF signaling / angiogenesis            0.650                 0.98 

In [31]:
REFERENCE_FONDAPARINUX = {
    "name": "Fondaparinux",
    "smiles": "O[C@H]1[C@H](O)[C@@H](N=SOO[O-])[C@@H](O[C@@H]([C@H](F)[C@@H](O)[C@H](O[C@@H]([C@H](OSOO[O-])[C@@H](N=SOO[O-])[C@@H](O[C@@H]([C@H](O)[C@H]2OSOO[O-])[C@H](C(O)=O)O[C@H]2O[C@H]3C(O)[C@@H](N=SOO[O-])[C@@H](OC)O[C@@H]3COS(=O)([O-])=O)O4)[C@H]4COS(=O)([O-])=O)O5)[C@H]5C(O)=O)O[C@@H]1COS(=O)([O-])=O",
    "description": """
    Fondaparinux is a synthetic sulfated pentasaccharide with strong antithrombin III binding
    and selective anti-factor Xa anticoagulant activity.
    """,
    "num_sugars": 5,
    "n_sulfate": 2,
    "o2_sulfate": 1,
    "o3_sulfate": 1,
    "o6_sulfate": 3,
    "carboxylates": 2,
    "has_fondaparinux_like_ATIII_motif": True,
    "has_fluorination": False,
    "has_hydrophobic_tag": False,
    "known_anti_Xa_activity": True
}

In [32]:
compound_x_df, compound_x_hs, compound_x_smiles = predict_targets(COMPOUND_X, HS_BINDING_PROTEINS)
fondaparinux_df, fondaparinux_hs, fondaparinux_smiles = predict_targets(REFERENCE_FONDAPARINUX, HS_BINDING_PROTEINS)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
comparison_df = compound_x_df.merge(
    fondaparinux_df,
    on=["Protein", "Gene", "Predicted application"],
    suffixes=("_Compound_X", "_Fondaparinux")
)

comparison_df["Delta_vs_Fondaparinux"] = (
    comparison_df["Final priority score_Compound_X"] -
    comparison_df["Final priority score_Fondaparinux"]
)

comparison_df["Interpretation"] = comparison_df["Delta_vs_Fondaparinux"].apply(
    lambda x: "Compound X enriched vs fondaparinux" if x > 0.05
    else "Fondaparinux-like" if abs(x) <= 0.05
    else "Lower than fondaparinux"
)

In [34]:
with pd.ExcelWriter("Compound_X_vs_Fondaparinux_HS_Probe.xlsx", engine="openpyxl") as writer:
    compound_x_df.to_excel(writer, sheet_name="Compound_X_Targets", index=False)
    fondaparinux_df.to_excel(writer, sheet_name="Fondaparinux_Targets", index=False)
    comparison_df.to_excel(writer, sheet_name="Comparative_Targets", index=False)